##  Flight Fare Prediction Project

### Problem Statement
Flight ticket prices are highly volatile and unpredictable. This project uses machine learning to analyze historical flight data and build a predictive model to estimate future flight prices, helping customers plan their journeys and airlines optimize their pricing strategies.

In [1]:
import pandas as pd
import numpy as np

In [2]:
# Load the dataset
df = pd.read_excel("Flight_Fare.xlsx")

In [3]:
# Display the first 5 rows
print("--- First 5 Rows of Data ---")
display(df.head())

--- First 5 Rows of Data ---


,Airline,Date_of_Journey,Source,Destination,Route,Dep_Time,Arrival_Time,Duration,Total_Stops,Additional_Info,Price
0,IndiGo,24/03/2019,Banglore,New Delhi,BLR → DEL,22:20,01:10 22 Mar,2h 50m,non-stop,No info,3897
1,Air India,1/05/2019,Kolkata,Banglore,CCU → IXR → BBI → BLR,05:50,13:15,7h 25m,2 stops,No info,7662
2,Jet Airways,9/06/2019,Delhi,Cochin,DEL → LKO → BOM → COK,09:25,04:25 10 Jun,19h,2 stops,No info,13882
3,IndiGo,12/05/2019,Kolkata,Banglore,CCU → NAG → BLR,18:05,23:30,5h 25m,1 stop,No info,6218
4,IndiGo,01/03/2019,Banglore,New Delhi,BLR → NAG → DEL,16:50,21:35,4h 45m,1 stop,No info,13302


In [4]:
# Check basic information about the columns
print("\n--- Data Information ---")
df.info()


--- Data Information ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10683 entries, 0 to 10682
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Airline          10683 non-null  object
 1   Date_of_Journey  10683 non-null  object
 2   Source           10683 non-null  object
 3   Destination      10683 non-null  object
 4   Route            10682 non-null  object
 5   Dep_Time         10683 non-null  object
 6   Arrival_Time     10683 non-null  object
 7   Duration         10683 non-null  object
 8   Total_Stops      10682 non-null  object
 9   Additional_Info  10683 non-null  object
 10  Price            10683 non-null  int64 
dtypes: int64(1), object(10)
memory usage: 918.2+ KB


In [5]:
# Check for missing values
print("\n--- Missing Values Count ---")
print(df.isnull().sum())


--- Missing Values Count ---
Airline            0
Date_of_Journey    0
Source             0
Destination        0
Route              1
Dep_Time           0
Arrival_Time       0
Duration           0
Total_Stops        1
Additional_Info    0
Price              0
dtype: int64


In [6]:
# Step 2.1: Drop the single missing row
df.dropna(inplace=True)

In [7]:
# Step 2.2: Unify 'New Delhi' and 'Delhi'
df['Destination'] = df['Destination'].replace('New Delhi', 'Delhi')

In [8]:
# Step 2.3: Drop Route and Additional_Info columns
df.drop(columns=['Route', 'Additional_Info'], inplace=True)

In [9]:
# Verify the changes
print("--- Remaining Missing Values ---")
print(df.isnull().sum())

--- Remaining Missing Values ---
Airline            0
Date_of_Journey    0
Source             0
Destination        0
Dep_Time           0
Arrival_Time       0
Duration           0
Total_Stops        0
Price              0
dtype: int64


In [10]:
print("\n--- Unique Destinations After Unification ---")
print(df['Destination'].unique())


--- Unique Destinations After Unification ---
['Delhi' 'Banglore' 'Cochin' 'Kolkata' 'Hyderabad']


In [11]:
print("\n--- Current Columns Left ---")
print(df.columns.tolist())


--- Current Columns Left ---
['Airline', 'Date_of_Journey', 'Source', 'Destination', 'Dep_Time', 'Arrival_Time', 'Duration', 'Total_Stops', 'Price']


In [12]:
# Step 3.1: Convert Date_of_Journey to Day and Month
df['Journey_Day'] = pd.to_datetime(df['Date_of_Journey'], format='%d/%m/%Y').dt.day
df['Journey_Month'] = pd.to_datetime(df['Date_of_Journey'], format='%d/%m/%Y').dt.month
df.drop(columns=['Date_of_Journey'], inplace=True)

In [13]:
# Step 3.2: Extract Hour and Minute from Departure Time
df['Dep_Hour'] = pd.to_datetime(df['Dep_Time'], format='%H:%M').dt.hour
df['Dep_Min'] = pd.to_datetime(df['Dep_Time'], format='%H:%M').dt.minute
df.drop(columns=['Dep_Time'], inplace=True)

In [14]:
# Extract Hour and Minute from Arrival Time (handling the extra date text if present)
df['Arrival_Hour'] = pd.to_datetime(df['Arrival_Time'].str.split(' ').str[0], format='%H:%M').dt.hour
df['Arrival_Min'] = pd.to_datetime(df['Arrival_Time'].str.split(' ').str[0], format='%H:%M').dt.minute
df.drop(columns=['Arrival_Time'], inplace=True)

In [15]:
# Step 3.3: Function to convert flight duration into total minutes
def duration_to_mins(duration):
    hours, mins = 0, 0
    for part in duration.split():
        if 'h' in part:
            hours = int(part.replace('h', ''))
        elif 'm' in part:
            mins = int(part.replace('m', ''))
    return hours * 60 + mins


df['Duration_Mins'] = df['Duration'].apply(duration_to_mins)
df.drop(columns=['Duration'], inplace=True)

In [16]:
# Step 3.4: Map 'Total_Stops' text to numerical integer labels
stops_map = {
    'non-stop': 0,
    '1 stop': 1,
    '2 stops': 2,
    '3 stops': 3,
    '4 stops': 4,
}
df['Total_Stops'] = df['Total_Stops'].map(stops_map)

In [17]:
# Take a look at our newly engineered numerical dataset
print("--- Features After Conversion ---")
display(df.head())

--- Features After Conversion ---


,Airline,Source,Destination,Total_Stops,Price,Journey_Day,Journey_Month,Dep_Hour,Dep_Min,Arrival_Hour,Arrival_Min,Duration_Mins
0,IndiGo,Banglore,Delhi,0,3897,24,3,22,20,1,10,170
1,Air India,Kolkata,Banglore,2,7662,1,5,5,50,13,15,445
2,Jet Airways,Delhi,Cochin,2,13882,9,6,9,25,4,25,1140
3,IndiGo,Kolkata,Banglore,1,6218,12,5,18,5,23,30,325
4,IndiGo,Banglore,Delhi,1,13302,1,3,16,50,21,35,285


In [18]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

In [19]:
# Step 4.1: Split data into independent features (X) and target variable (y)
X = df.drop(columns=['Price'])
y = df['Price']

In [20]:
# Step 4.2: Use an 80/20 Train-Test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [21]:
# Step 4.3: Set up a preprocessor to One-Hot Encode our categorical columns
categorical_cols = ['Airline', 'Source', 'Destination']

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
    ], 
    remainder='passthrough'
)

In [22]:
# Apply the preprocessing transformation to our training and testing data
X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)

In [23]:
# Get names of our newly created One-Hot columns to see what happened
encoded_feature_names = preprocessor.get_feature_names_out()
X_train_df = pd.DataFrame(X_train_encoded, columns=encoded_feature_names)

print(f"Original feature count: {X_train.shape[1]} columns")
print(f"Encoded feature count: {X_train_df.shape[1]} columns")
print("\n--- Snippet of the processed matrix ready for Machine Learning ---")
display(X_train_df.head())

Original feature count: 11 columns
Encoded feature count: 30 columns

--- Snippet of the processed matrix ready for Machine Learning ---


,cat__Airline_Air Asia,cat__Airline_Air India,cat__Airline_GoAir,cat__Airline_IndiGo,cat__Airline_Jet Airways,cat__Airline_Jet Airways Business,cat__Airline_Multiple carriers,cat__Airline_Multiple carriers Premium economy,cat__Airline_SpiceJet,cat__Airline_Trujet,...,cat__Destination_Hyderabad,cat__Destination_Kolkata,remainder__Total_Stops,remainder__Journey_Day,remainder__Journey_Month,remainder__Dep_Hour,remainder__Dep_Min,remainder__Arrival_Hour,remainder__Arrival_Min,remainder__Duration_Mins
0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,27.0,5.0,8.0,30.0,19.0,15.0,645.0
1,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,9.0,5.0,11.0,30.0,12.0,35.0,1505.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,1.0,24.0,4.0,15.0,45.0,22.0,5.0,380.0
3,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,21.0,3.0,12.0,50.0,1.0,35.0,765.0
4,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,2.0,24.0,6.0,17.0,15.0,19.0,15.0,1560.0


In [24]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor

In [25]:
# Step 5.1: Initialize the models with a fixed random_state for reproducible results
models = {
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Extra Trees': ExtraTreesRegressor(n_estimators=100, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42)
}

In [26]:
# A dictionary to temporarily store the calculated predictions for evaluation in the next phase
predictions = {}

In [27]:
# Step 5.2 & 5.3: Loop through, train, and test each model automatically
print("--- Training Models... Please wait a few seconds ---")
for name, model in models.items():
    # Train the model on the encoded training data
    model.fit(X_train_encoded, y_train)
    
    # Generate predictions on the encoded test data
    y_pred = model.predict(X_test_encoded)
    
    # Save predictions for the final evaluation phase
    predictions[name] = y_pred
    print(f"✓ {name} has completed training and generated predictions.")

--- Training Models... Please wait a few seconds ---
✓ Decision Tree has completed training and generated predictions.
✓ Extra Trees has completed training and generated predictions.
✓ Random Forest has completed training and generated predictions.


In [28]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [29]:
# A dictionary to hold the finalized scores for all algorithms
evaluation_results = {}

In [30]:
# Calculate scores for each model based on the predictions stored in Phase 5
for name, y_pred in predictions.items():
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    
    evaluation_results[name] = {
        'R2 Score (Accuracy)': f"{r2:.4f}",
        'MAE (Avg Error)': f"₹{mae:.2f}",
        'RMSE': f"₹{rmse:.2f}"
    }

In [35]:
# Convert results into a beautiful DataFrame summary table
results_df = pd.DataFrame(evaluation_results).T
print("Final Model Performance Comparison Table")
display(results_df)

Final Model Performance Comparison Table


,R2 Score (Accuracy),MAE (Avg Error),RMSE
Decision Tree,0.7766,₹1293.86,₹2194.71
Extra Trees,0.8032,₹1229.88,₹2060.06
Random Forest,0.8171,₹1171.04,₹1985.71


### Project Conclusions & Technical Report

#### 1. Challenges Faced & Techniques Used
* **Challenge 1: Complex Time Strings**
  * *Context:* Columns like `Arrival_Time` and `Dep_Time` contained mixed text formats (e.g., `"01:10 22 Mar"`).
  * *Technique:* Used Pandas datetime processing to split them into separate numerical columns (`Hour` and `Minute`) so the model could run math equations on them.
* **Challenge 2: Irregular Flight Durations**
  * *Context:* The `Duration` column was inconsistent, mixing formats like `"2h 50m"`, `"19h"`, and `"45m"`[cite: 1].
  * *Technique:* Created a custom Python splitting function to convert all text durations into a single, uniform number of total minutes (`Duration_Mins`).
* **Challenge 3: Duplicate Location Names**
  * *Context:* The `Destination` column contained both `"Delhi"` and `"New Delhi"`, which is the same market.
  * *Technique:* Unified them into a single `"Delhi"` string to clean up the data and prevent errors during One-Hot Encoding.

#### 2. Production Model Recommendation
* **Recommended Model:** Random Forest Regressor
* **Performance:** It achieved the highest overall **$R^2$ Score of 81.71%** and the lowest average error (**MAE of ₹1,171.04**). 
* **Reasoning:** By combining an ensemble of multiple decision trees, Random Forest reduces overfitting and gives the most stable price predictions on unseen data.